# ED Pipeline — v6→v8 Merge Notebook (Phase-2 Guarded)

Canonical v8 SHA-256: `8a54a37cbeb4d31ac543d0003964b10a9bf9a073ccb47bc41e37e3625af7e20a`


In [43]:
# RERUN BOOTSTRAP (safe, append-only)
try:
    CONFIG
except NameError:
    CONFIG = {}

CONFIG.setdefault("DATA_ROOT", "/mnt/data")
CONFIG.setdefault("EVENT_LOG_PATH", f"{CONFIG['DATA_ROOT'].rstrip('/')}/event_log.jsonl")

CONFIG["RUN_PIPELINE"] = True
CONFIG["RUN_UI"] = True

print("RUN_PIPELINE:", CONFIG["RUN_PIPELINE"])
print("RUN_UI:", CONFIG["RUN_UI"])
print("EVENT_LOG_PATH:", CONFIG["EVENT_LOG_PATH"])


RUN_PIPELINE: True
RUN_UI: True
EVENT_LOG_PATH: /mnt/data/event_log.jsonl


In [44]:
try: CONFIG
except NameError: CONFIG={}
print("RUN_PIPELINE:", CONFIG.get("RUN_PIPELINE"))
print("RUN_UI:", CONFIG.get("RUN_UI"))
print("Has bundle:", "phase2_bundle" in globals())
print("Has HL7 helper:", "process_hl7_and_log" in globals())
print("Event log:", CONFIG.get("EVENT_LOG_PATH"))


RUN_PIPELINE: True
RUN_UI: True
Has bundle: False
Has HL7 helper: False
Event log: /mnt/data/event_log.jsonl


In [45]:
# Safe guard flip (creates CONFIG if missing)
try:
    CONFIG
except NameError:
    CONFIG = {}

CONFIG["RUN_PIPELINE"] = True
print("RUN_PIPELINE:", CONFIG["RUN_PIPELINE"])


RUN_PIPELINE: True


# ED Ops Pipeline — v6 (Full, single-file)
All core services inlined here.

In [46]:
import os, sys
from pathlib import Path
if "/mnt/data" not in sys.path: sys.path.insert(0, "/mnt/data")
try:
    CONFIG
except NameError:
    DATA_ROOT = os.environ.get("DATA_ROOT", "/mnt/data")
    CONFIG = {"DATA_ROOT": DATA_ROOT}
defaults = {
    "EQUIPMENT_STATUS_PATH": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "equipment_status.csv"),
    "EQUIPMENT_MOVES_LOG_PATH": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "equipment_moves.csv"),
    "SOP_REGISTRY_PATH": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "sop_registry.csv"),
    "QR_OUTPUT_DIR": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "qr"),
    "EVENT_LOG_PATH": str(Path(CONFIG.get("DATA_ROOT","/mnt/data")) / "event_log.jsonl"),
    "RUN_UI": False,
    "RUN_PIPELINE": False,
}
CONFIG.update({k: CONFIG.get(k, v) for k, v in defaults.items()})
RUN_UI = CONFIG["RUN_UI"]; RUN_PIPELINE = CONFIG["RUN_PIPELINE"]
for k in ["QR_OUTPUT_DIR","EVENT_LOG_PATH","SOP_REGISTRY_PATH","EQUIPMENT_STATUS_PATH","EQUIPMENT_MOVES_LOG_PATH"]:
    p = Path(CONFIG[k]); (p.parent if p.suffix else p).mkdir(parents=True, exist_ok=True)
print("bootstrap ready")

bootstrap ready


In [47]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List
import pandas as pd


In [48]:

@dataclass
class WorkflowState:
    encounter_id: Optional[str] = None
    patient_id: Optional[str] = None
    pending_orders: set = field(default_factory=set)
    completed_studies: set = field(default_factory=set)
    active_consults: set = field(default_factory=set)
    last_vitals_ts: Optional[pd.Timestamp] = None
    chest_pain: bool = False
    trauma: bool = False
    # context
    backlog_ct: int = 0
    backlog_lab: int = 0
    backlog_ecg: int = 0
    hour: int = 12
    role: str = "nurse"

def skill_need_ecg(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if state.chest_pain and ("ORDER_ECG" not in state.pending_orders) and ("ORDER_ECG" not in state.completed_studies):
        return {"action":"ORDER_ECG", "reason":"Chest pain without ECG", "urgency":"high"}
    return None

def skill_abnormal_ecg_no_consult(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_ECG" in state.completed_studies) and ("ECG_ABNORMAL" in state.completed_studies) and ("CARDIOLOGY" not in state.active_consults):
        return {"action":"PAGE_CARDIOLOGY", "reason":"Abnormal ECG without consult", "urgency":"high"}
    return None

def skill_ct_delayed(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if ("ORDER_CT" in state.pending_orders) and ("CT_RESULT" not in state.completed_studies):
        return {"action":"FOLLOW_UP_IMAGING", "reason":"CT pending > 60m", "urgency":"medium"}
    return None

def skill_pending_labs_deteriorating(state: WorkflowState) -> Optional[Dict[str,Any]]:
    if (("LAB_TROPONIN" in state.pending_orders) or ("LAB_PANEL" in state.pending_orders)) and ("Deteriorating" in state.completed_studies):
        return {"action":"EXPEDITE_LABS", "reason":"Pending labs + deterioration", "urgency":"high"}
    return None

SKILLS = [
    skill_need_ecg,
    skill_abnormal_ecg_no_consult,
    skill_ct_delayed,
    skill_pending_labs_deteriorating,
]

def generate_candidates(state: WorkflowState) -> List[Dict[str,Any]]:
    out = []
    for s in SKILLS:
        r = s(state)
        if r: out.append(r)
    return out[:5]


In [49]:
from __future__ import annotations
from typing import List, Dict, Any, Tuple
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
class TinyCritics:
    def __init__(self):
        base = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),("clf", LogisticRegression(max_iter=1000))])
        self.model = CalibratedClassifierCV(base, method="isotonic", cv=3)
        self.num_features_: List[str] = ["hour","spo2","backlog_ct","backlog_lab","backlog_ecg","pending_n","completed_n","consults_n","since_vitals_min"]
        self.cat_features_: List[str] = ["role","cp","resp","trauma"]
        self.preproc = ColumnTransformer([("num", SimpleImputer(strategy="median"), self.num_features_),("cat", OneHotEncoder(handle_unknown="ignore"), self.cat_features_)], remainder="drop")
        self.is_fit = False
    def _featurize(self, X: List[Dict[str,Any]]) -> pd.DataFrame:
        rows = []
        for x in X:
            s = x.get("state"); a = x.get("action", {})
            if hasattr(s, "feature_dict"): f = s.feature_dict()
            elif isinstance(s, dict): f = dict(s)
            else: f = {}
            f["action_label"] = str(a.get("label") or a.get("id") or "action")
            rows.append(f)
        df = pd.DataFrame(rows)
        for col in self.num_features_ + self.cat_features_:
            if col not in df.columns: df[col] = np.nan if col in self.num_features_ else "NA"
        return df[self.num_features_ + self.cat_features_ + ["action_label"]]
    def fit(self, samples: List[Dict[str,Any]], y: np.ndarray) -> "TinyCritics":
        df = self._featurize(samples)
        Xp = self.preproc.fit_transform(df[self.num_features_ + self.cat_features_]); self.model.fit(Xp, y); self.is_fit = True; return self
    def score(self, state, actions: List[Dict[str,Any]]):
        X = self._featurize([{"state": state, "action": a} for a in actions])
        if not self.is_fit:
            n = len(actions); return np.full(n, 0.5), np.zeros(n), np.zeros(n)
        Xp = self.preproc.transform(X[self.num_features_ + self.cat_features_])
        p = self.model.predict_proba(Xp)[:, 1]
        benefit = (1.0 - np.clip(X["backlog_ct"].fillna(0), 0, 10)/10.0).to_numpy()
        burden = (np.clip(X["since_vitals_min"].fillna(60), 0, 120)/120.0).to_numpy()
        return p, benefit, burden
print("TinyCritics ready")

TinyCritics ready


In [50]:
def rule_hs_tnt(value):
    try: v = float(value)
    except Exception: return 0.50
    if v < 14: return 0.50
    if 14 <= v <= 51: return 0.20
    return 0.50
assert rule_hs_tnt(13.9)==0.50 and rule_hs_tnt(14.0)==0.20 and rule_hs_tnt(51.0)==0.20 and rule_hs_tnt(51.1)==0.50
print("troponin_rules ok")

troponin_rules ok


In [51]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from pathlib import Path
import pandas as pd, numpy as np
def _cfg(CONFIG: Any, key: str, default: Any=None) -> Any:
    try: return CONFIG.get(key, default)
    except Exception: return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default
def _ensure_parent(p: Path): p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)
@dataclass
class EquipmentRecord:
    equip_id: str; name: str=""; location: str=""; status: str=""; last_seen: Optional[str]=None; battery: Optional[float]=None; confidence: Optional[float]=None
    def to_row(self)->Dict[str,Any]: return {"equip_id":self.equip_id,"name":self.name,"location":self.location,"status":self.status,"last_seen":self.last_seen,"battery":self.battery,"confidence":self.confidence}
class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv=Path(status_csv); _ensure_parent(self.status_csv)
        if not self.status_csv.exists(): pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"]).to_csv(self.status_csv, index=False)
    def read(self)->pd.DataFrame:
        try: df=pd.read_csv(self.status_csv); 
        except Exception: return pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"])
        if "equip_id" in df.columns: df["equip_id"]=df["equip_id"].astype(str); return df
    def upsert(self, rec: EquipmentRecord)->None:
        df=self.read(); row=pd.DataFrame([rec.to_row()])
        if df.empty: df=row
        else:
            mask=(df["equip_id"].astype(str)==str(rec.equip_id))
            if mask.any(): df.loc[mask,:]=row.values
            else: df=pd.concat([df,row], ignore_index=True)
        df.to_csv(self.status_csv, index=False)
class MovesLogRepository:
    def __init__(self, moves_csv: Path):
        self.moves_csv=Path(moves_csv); _ensure_parent(self.moves_csv)
        if not self.moves_csv.exists(): pd.DataFrame(columns=["equip_id","from","to","ts"]).to_csv(self.moves_csv, index=False)
    def append(self, equip_id:str, loc_from:str, loc_to:str, ts_iso:str)->None:
        row=pd.DataFrame([{"equip_id":equip_id,"from":loc_from,"to":loc_to,"ts":ts_iso}])
        try: prev=pd.read_csv(self.moves_csv) if self.moves_csv.exists() else None; df=pd.concat([prev,row], ignore_index=True) if prev is not None else row
        except Exception: df=row
        df.to_csv(self.moves_csv, index=False)
    def read(self)->pd.DataFrame:
        try: return pd.read_csv(self.moves_csv)
        except Exception: return pd.DataFrame(columns=["equip_id","from","to","ts"])
class SOPRegistry:
    def __init__(self, sop_csv: Path): self.sop_csv=Path(sop_csv); _ensure_parent(self.sop_csv)
    def read(self)->pd.DataFrame:
        if self.sop_csv.exists():
            try:
                df=pd.read_csv(self.sop_csv)
                for col in ["sop_id","title","pdf_path"]:
                    if col not in df.columns: df[col]=""
                return df
            except Exception: pass
        return pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])
class QRService:
    def __init__(self,out_dir:Path): 
        self.out_dir=Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self,payload:str)->str:
        try:
            import qrcode
            fp=self.out_dir/f"qr_{abs(hash(payload))}.png"
            img=qrcode.make(payload); img.save(fp); return str(fp)
        except Exception: return f"[QR fallback] {payload}"
    def decode_file(self, image_bytes:bytes):
        try:
            from PIL import Image; import io
            img=Image.open(io.BytesIO(image_bytes))
            try:
                from pyzbar.pyzbar import decode as zbar_decode
                res=zbar_decode(img); 
                if res: return res[0].data.decode("utf-8","ignore")
            except Exception: pass
        except Exception: pass
        return None
class TrackerService:
    def __init__(self, equipment_repo:EquipmentRepository, moves_repo:MovesLogRepository, sop_registry:SOPRegistry, qr:QRService, config:Any):
        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config
    @classmethod
    def from_config(cls, CONFIG:Any)->"TrackerService":
        return cls(EquipmentRepository(Path(_cfg(CONFIG,"EQUIPMENT_STATUS_PATH"))),
                   MovesLogRepository(Path(_cfg(CONFIG,"EQUIPMENT_MOVES_LOG_PATH"))),
                   SOPRegistry(Path(_cfg(CONFIG,"SOP_REGISTRY_PATH"))),
                   QRService(Path(_cfg(CONFIG,"QR_OUTPUT_DIR"))), CONFIG)
    def equipment_status(self)->pd.DataFrame: return self.equipment_repo.read()
    def log_move(self, equip_id:str, loc_from:str, loc_to:str)->None:
        ts_iso=pd.Timestamp.utcnow().isoformat(); df=self.equipment_repo.read()
        row=df[df["equip_id"].astype(str)==str(equip_id)]; name=row["name"].iloc[0] if not row.empty and "name" in row.columns else ""
        rec=EquipmentRecord(equip_id=equip_id,name=name,location=loc_to,status="moved",last_seen=ts_iso)
        self.equipment_repo.upsert(rec); self.moves_repo.append(equip_id, loc_from or "", loc_to, ts_iso)
    def find_equipment(self, query:str)->pd.DataFrame:
        q=(query or "").strip().lower(); df=self.equipment_repo.read()
        if not q: return df
        def hit(r): return any(q in str(r.get(k,"")).lower() for k in ["equip_id","name","location","status"])
        return df[df.apply(hit, axis=1)]
    def overdue_equipment(self, threshold_minutes:int=120)->pd.DataFrame:
        df=self.equipment_repo.read().copy()
        if df.empty or "last_seen" not in df.columns: return df.iloc[0:0]
        ts=pd.to_datetime(df["last_seen"],errors="coerce",utc=True); age_min=(pd.Timestamp.utcnow().tz_localize("UTC")-ts).dt.total_seconds()/60.0
        df["age_min"]=age_min; return df[age_min>float(threshold_minutes)].sort_values("age_min", ascending=False)
    def movement_stats(self)->Dict[str,pd.DataFrame]:
        log=self.moves_repo.read()
        if log.empty: return {"moves_per_equipment":log,"routes":log}
        per_eq=log.groupby("equip_id").size().reset_index(name="moves").sort_values("moves", ascending=False)
        routes=log.groupby(["from","to"]).size().reset_index(name="count").sort_values("count", ascending=False)
        return {"moves_per_equipment":per_eq,"routes":routes}
    def sop_table(self)->pd.DataFrame: return self.sop_registry.read()
    def search_sop(self, query:str)->pd.DataFrame:
        df=self.sop_registry.read().copy(); q=(query or "").strip().lower()
        if df.empty or not q: return df
        cols=[c for c in ["sop_id","title","keywords","version","status"] if c in df.columns]
        mask=df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)
        return df[mask]
    def make_qr(self,payload:str)->str: return self.qr.make(payload)
    def decode_qr_bytes(self, image_bytes:bytes): return self.qr.decode_file(image_bytes)
print("tracker core ready")

tracker core ready


In [52]:
from pathlib import Path
from typing import Any, Dict, List
def _slugify(text:str)->str:
    import re; s=re.sub(r"[^a-zA-Z0-9]+","-",text.strip().lower()).strip("-"); return s or "sop"
def refresh_sop_registry(CONFIG: Any, base_url: str="https://sop-notaufnahme.de/sop/")->Dict[str,Any]:
    out_csv=Path(CONFIG["SOP_REGISTRY_PATH"]); pdf_dir=Path(CONFIG["DATA_ROOT"])/"sop_pdfs"; pdf_dir.mkdir(parents=True, exist_ok=True)
    try:
        import requests; from bs4 import BeautifulSoup
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":f"missing libs: {e}"}
    found=saved=errors=0; items=[]
    try:
        r=requests.get(base_url, timeout=15); r.raise_for_status(); soup=BeautifulSoup(r.text,"html.parser")
        links=sorted({a["href"] for a in soup.find_all("a", href=True) if "/product/" in a["href"] and a["href"].startswith("http")})
        for url in links:
            try:
                pr=requests.get(url, timeout=15); pr.raise_for_status(); ps=BeautifulSoup(pr.text,"html.parser")
                ttag=ps.find(["h1","h2"]); title=ttag.get_text(strip=True) if ttag else (ps.find("title").get_text(strip=True) if ps.find("title") else url)
                pdfs=[a["href"] for a in ps.find_all("a", href=True) if a["href"].lower().endswith(".pdf")]
                pdf_url=pdfs[0] if pdfs else None; sop_id=_slugify(title or url.split("/")[-2]); pdf_path=""
                if pdf_url:
                    try:
                        fn=sop_id+".pdf"; outp=pdf_dir/fn
                        with requests.get(pdf_url, stream=True, timeout=30) as dr:
                            dr.raise_for_status()
                            with open(outp,"wb") as f:
                                for chunk in dr.iter_content(8192):
                                    if chunk: f.write(chunk)
                        pdf_path=str(outp); saved+=1
                    except Exception:
                        errors+=1; pdf_path=pdf_url
                items.append({"sop_id":sop_id,"title":title or sop_id,"pdf_path":pdf_path,"version":"","status":"fetched" if pdf_path else "linked","keywords":"","checklist":"","source_url":url})
                found+=1
            except Exception: errors+=1; continue
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":str(e)}
    import pandas as pd
    try:
        if out_csv.exists(): df=pd.read_csv(out_csv)
        else: df=pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])
        df=df.copy()
        if df.empty: new_df=pd.DataFrame(items)
        else:
            df["sop_id"]=df["sop_id"].astype(str)
            for i in items:
                mask=(df["sop_id"]==str(i["sop_id"]))
                if mask.any():
                    for k,v in i.items():
                        if k in df.columns and (pd.isna(df.loc[mask,k]).all() or str(df.loc[mask,k].iloc[0]).strip()=="" or k in ["pdf_path","status","source_url"]):
                            df.loc[mask,k]=v
                else:
                    df=pd.concat([df, pd.DataFrame([i])], ignore_index=True)
            new_df=df
        new_df.to_csv(out_csv, index=False)
    except Exception as e:
        errors+=1
    return {"found":found,"saved":saved,"errors":errors,"csv":str(out_csv),"dir":str(pdf_dir)}
def load_priority_flows(json_path:str)->Dict[str,Any]:
    import json
    try:
        with open(json_path,"r",encoding="utf-8") as f: return json.load(f)
    except Exception: return {}
print("sop auto ready")

sop auto ready


In [53]:
import importlib
def _try_import(name:str):
    try: return importlib.import_module(name)
    except Exception: return None
def run_icu_constraints(state):
    if not RUN_PIPELINE: return {}
    mod = _try_import("icu_constraints") or _try_import("modeling_icu_constraints")
    if mod and hasattr(mod,"compute_icu_flags"):
        try: return dict(mod.compute_icu_flags(state))
        except Exception: return {}
    return {}
def mesh_route_actions(state, actions):
    if not RUN_PIPELINE: return actions
    mod = _try_import("agent_mesh") or _try_import("ed_agent_mesh")
    if mod and hasattr(mod,"route"):
        try: return list(mod.route(state, actions))
        except Exception: return actions
    return actions
def trainer_fit_critic(critic, samples, y):
    if not RUN_PIPELINE: return critic
    mod = _try_import("trainer") or _try_import("ed_trainer")
    if mod and hasattr(mod,"fit_critic"):
        try: return mod.fit_critic(critic, samples, y)
        except Exception: return critic
    return critic
print("phase2 bridge ready")

phase2 bridge ready


In [54]:
# --- [HOTFIX-2] Force ipywidgets fallback for run_ui (no Streamlit) ---
from IPython.display import display, clear_output
import ipywidgets as W

def _fallback_run_ui(tracker, get_state, get_actions, critic):
    box = W.Output(layout=W.Layout(border="1px solid #ddd", padding="6px"))
    btn_refresh = W.Button(description="Refresh")
    btn_actions = W.Button(description="Suggest Actions", button_style="info")

    def do_refresh(_):
        with box:
            clear_output()
            try:
                state = get_state()
                print("Operational dashboard (fallback UI)")
                df = state.get("status_df") if isinstance(state, dict) else None
                if df is not None:
                    try:
                        import pandas as pd  # optional
                        display(df.head())
                    except Exception:
                        print(df.head() if hasattr(df, "head") else df)
                else:
                    print(state)
            except Exception as e:
                print("state error:", e)

    def do_actions(_):
        with box:
            clear_output()
            try:
                acts = get_actions()
                print("Action candidates:")
                print(acts)
            except Exception as e:
                print("actions error:", e)

    btn_refresh.on_click(do_refresh)
    btn_actions.on_click(do_actions)
    display(W.VBox([W.HBox([btn_refresh, btn_actions]), box]))
    print("[ui] Streamlit bypassed → ipywidgets fallback active.")

# **Override** any prior definition unconditionally
run_ui = _fallback_run_ui


In [55]:
# --- [HOTFIX] Streamlit-free run_ui (guarded) ---
if CONFIG.get("RUN_UI", False):
    try:
        import streamlit  # probe only
        _has_streamlit = True
    except Exception:
        _has_streamlit = False

    if not _has_streamlit:
        from IPython.display import display, clear_output
        import ipywidgets as W

        def run_ui(tracker, get_state, get_actions, critic):
            box = W.Output(layout=W.Layout(border="1px solid #ddd", padding="6px"))
            btn_refresh = W.Button(description="Refresh")
            btn_actions = W.Button(description="Suggest Actions", button_style="info")

            def do_refresh(_):
                with box:
                    clear_output()
                    try:
                        state = get_state()
                        print("Operational dashboard (fallback UI)")
                        df = state.get("status_df") if isinstance(state, dict) else None
                        if df is not None:
                            try:
                                import pandas as pd
                                display(df.head())
                            except Exception:
                                print(df.head() if hasattr(df, "head") else df)
                        else:
                            print(state)
                    except Exception as e:
                        print("state error:", e)

            def do_actions(_):
                with box:
                    clear_output()
                    try:
                        acts = get_actions()
                        print("Action candidates:")
                        print(acts)
                    except Exception as e:
                        print("actions error:", e)

            btn_refresh.on_click(do_refresh)
            btn_actions.on_click(do_actions)
            display(W.VBox([W.HBox([btn_refresh, btn_actions]), box]))
            print("[ui] Streamlit not available → using ipywidgets fallback.")


In [56]:
def run_ui(tracker, get_state, get_actions, critic):
    import streamlit as st, pandas as pd, numpy as np
    st.set_page_config(page_title="ED Tracker — Full", layout="wide")
    st.title("ED Tracker — Core Ops (Full)")
    c0, c1, c2, c3 = st.columns([2,2,2,2])
    with c0:
        thresh = st.number_input("Overdue threshold (min)", min_value=5, max_value=720, value=120, step=5)
    with c1:
        if st.button("Refresh"): st.experimental_rerun()
    st.header("Equipment")
    eq_df = tracker.equipment_status()
    s1, s2 = st.columns([2,1])
    with s1:
        q = st.text_input("Find equipment (ID / name / location / status)", "")
        filt = tracker.find_equipment(q) if q else eq_df
        st.dataframe(filt, use_container_width=True, height=260)
    with s2:
        overdue = tracker.overdue_equipment(int(thresh))
        st.subheader("Overdue")
        if overdue.empty: st.write("None")
        else: st.dataframe(overdue[["equip_id","name","location","last_seen","age_min"]], use_container_width=True, height=200)
    st.markdown("**Update location / log move**")
    mc1, mc2, mc3, mc4 = st.columns([2,2,2,1])
    with mc1: sel_id = st.selectbox("Equipment ID", [""] + sorted(list(eq_df.get("equip_id", []))))
    with mc2: loc_from = st.text_input("From", "")
    with mc3: loc_to = st.text_input("To", "")
    with mc4:
        if st.button("Log move") and sel_id and loc_to:
            tracker.log_move(sel_id, loc_from, loc_to); st.success(f"Move logged: {sel_id} → {loc_to}")
    st.header("QR")
    qr_col1, qr_col2 = st.columns([2,2])
    with qr_col1:
        qr_txt = st.text_input("QR payload to generate", "")
        if st.button("Generate QR") and qr_txt:
            path = tracker.make_qr(qr_txt); st.write("QR saved to:", path)
    with qr_col2:
        st.write("Scan and update location")
        f = st.file_uploader("Upload QR image", type=["png","jpg","jpeg","webp"])
        manual_payload = st.text_input("Manual payload (fallback if decoding fails)", "")
        new_loc = st.text_input("New location (after scan)", "")
        if st.button("Scan & Update"):
            equip_payload = None
            if f is not None: equip_payload = tracker.decode_qr_bytes(f.read())
            if not equip_payload and manual_payload: equip_payload = manual_payload
            if equip_payload and new_loc:
                equip_id = equip_payload
                if "id=" in equip_payload:
                    try: equip_id = equip_payload.split("id=",1)[1].split("&",1)[0]
                    except Exception: equip_id = equip_payload
                tracker.log_move(str(equip_id), "", new_loc); st.success(f"Updated via payload. {equip_id} → {new_loc}")
            elif not new_loc: st.error("Provide a new location.")
            else: st.error("No QR payload detected (image or manual).")
    with st.expander("SOP auto-pull and flows", expanded=False):
        if st.button("Refresh SOPs from sop-notaufnahme.de"):
            res = refresh_sop_registry(CONFIG, base_url="https://sop-notaufnahme.de/sop/"); st.write(res)
        flows = load_priority_flows("/mnt/data/priority_flows.json")
        if flows:
            keys = sorted(list(flows.keys())); pickf = st.selectbox("Show flow", [""] + keys)
            if pickf:
                flow = flows[pickf]; st.subheader(flow.get("title", pickf))
                nodes = flow.get("nodes", []); edges = flow.get("edges", [])
                st.write("Nodes:", ", ".join([n.get("label", n.get("id","")) for n in nodes]))
                try:
                    import matplotlib.pyplot as plt
                    fig = plt.figure()
                    pos = {n["id"]:(i, 0) for i,n in enumerate(nodes)}
                    for n in nodes:
                        x,y = pos[n["id"]]; plt.scatter([x],[y]); plt.text(x,y+0.05,n.get("label", n["id"]), ha="center", rotation=45)
                    for a,b in edges:
                        xa,ya = pos.get(a,(0,0)); xb,yb = pos.get(b,(0,0)); plt.plot([xa,xb],[ya,yb])
                    plt.axis("off"); plt.title(flow.get("title", pickf)); st.pyplot(fig)
                except Exception: st.info("Graph display unavailable; showing list instead."); st.write(edges)
    st.header("SOPs")
    sop_q = st.text_input("Search SOPs (id/title/keywords)", "")
    sop_hits = tracker.search_sop(sop_q)
    if sop_hits.empty: st.info("No SOPs found.")
    else:
        st.dataframe(sop_hits[["sop_id","title","version","status"]], use_container_width=True, height=220)
        pick = st.selectbox("Open SOP", [""] + sop_hits["sop_id"].astype(str).tolist())
        if pick:
            row = sop_hits[sop_hits["sop_id"].astype(str)==pick].iloc[0]
            pdf = row.get("pdf_path","")
            if pdf: st.write("PDF path:", pdf)
            if "checklist" in sop_hits.columns and isinstance(row.get("checklist", None), str) and row["checklist"].strip():
                st.subheader("Checklist")
                steps = [s.strip() for s in row["checklist"].split("|") if s.strip()]
                completed = []
                for i, step in enumerate(steps, 1):
                    if st.checkbox(f"{i}. {step}", key=f"sop_{pick}_{i}"):
                        completed.append(i)
                st.caption(f"Completed {len(completed)}/{len(steps)} steps")
    st.header("Actions & Critic")
    state = get_state()
    if hasattr(state,"feature_dict"):
        feats = state.feature_dict(); since_v = feats.get("since_vitals_min", None)
        if since_v is not None:
            if since_v > 120: st.error(f"Lingering patient: since_vitals_min={since_v:.0f} > 120")
            else: st.success(f"Vitals recently checked: {since_v:.0f} min")
    if st.button("Mark vitals now") and hasattr(state,"touch_now"):
        state.touch_now(pd.Timestamp.utcnow()); st.success("Vitals timestamp updated.")
    actions = get_actions(state)
    if not actions: st.info("No actions available."); return
    p, benefit, burden = critic.score(state, actions)
    import pandas as pd, numpy as np
    view = pd.DataFrame({"id":[a.get("id") for a in actions],"label":[a.get("label") for a in actions],"p_accept":np.round(p,3),"benefit":np.round(benefit,3),"burden":np.round(burden,3)}).sort_values(["p_accept","benefit"], ascending=[False, False])
    st.dataframe(view, use_container_width=True, height=240)
    st.header("Equipment Movement Analytics")
    stats = tracker.movement_stats(); per_eq = stats["moves_per_equipment"]; routes = stats["routes"]
    if per_eq.empty: st.info("No movement data yet.")
    else:
        st.subheader("Moves per equipment"); st.dataframe(per_eq, use_container_width=True, height=240)
        try:
            import matplotlib.pyplot as plt
            fig = plt.figure(); x=per_eq["equip_id"].astype(str).tolist(); y=per_eq["moves"].tolist()
            plt.bar(x,y); plt.xticks(rotation=45, ha="right"); plt.title("Moves per Equipment"); st.pyplot(fig)
        except Exception: pass
        st.subheader("Top routes"); st.dataframe(routes, use_container_width=True, height=200)
print("ui ready")

ui ready


In [57]:
import pandas as pd
from pathlib import Path
E = Path(CONFIG["EQUIPMENT_STATUS_PATH"])
if not E.exists():
    pd.DataFrame([
        {"equip_id":"pump-001","name":"IV Pump","location":"A1","status":"ready","last_seen":pd.Timestamp.utcnow().isoformat(),"battery":0.9,"confidence":0.95},
        {"equip_id":"defib-002","name":"Defibrillator","location":"B2","status":"ready","last_seen":pd.Timestamp.utcnow().isoformat(),"battery":0.8,"confidence":0.90},
    ]).to_csv(E, index=False)
M = Path(CONFIG["EQUIPMENT_MOVES_LOG_PATH"])
if not M.exists(): pd.DataFrame(columns=["equip_id","from","to","ts"]).to_csv(M, index=False)
S = Path(CONFIG["SOP_REGISTRY_PATH"])
if not S.exists():
    sop_dir = Path(CONFIG["DATA_ROOT"]) / "sop_pdfs"; sop_dir.mkdir(parents=True, exist_ok=True)
    for i in range(1,4): (sop_dir / f"SOP_{i:02d}.pdf").write_bytes(b"%PDF-1.4\n% placeholder\n")
    pd.DataFrame([
        {"sop_id":"SOP_01","title":"Chest Pain Triage","pdf_path":str(sop_dir/"SOP_01.pdf"),"version":"0.1","status":"placeholder","keywords":"chest pain|ecg|troponin","checklist":"Open SOP|Order ECG|Record troponin|Reassess vitals"},
        {"sop_id":"SOP_02","title":"Sepsis Initial Bundle","pdf_path":str(sop_dir/"SOP_02.pdf"),"version":"0.1","status":"placeholder","keywords":"sepsis|qsofa|fluids","checklist":"Open SOP|Order labs|Start fluids|Antibiotics within 1h"},
        {"sop_id":"SOP_03","title":"Stroke Code","pdf_path":str(sop_dir/"SOP_03.pdf"),"version":"0.1","status":"placeholder","keywords":"stroke|nihs|ct","checklist":"Open SOP|CT head|Neurology consult|Thrombolysis criteria"},
    ]).to_csv(S, index=False)
print("seed done")

seed done


In [58]:
import pandas as pd, numpy as np
s=WorkflowState(role="nurse"); getattr(s,"touch_now",lambda *_:None)(pd.Timestamp.utcnow())
tc=TinyCritics(); p,b,u=tc.score(s,[{"id":"reassess_vitals","label":"Reassess vitals"},{"id":"order_ecg","label":"Order ECG"}])
assert len(p)==2 and (0<=p).all() and (p<=1).all()
from pathlib import Path
t=TrackerService.from_config(CONFIG)
_=t.equipment_status(); t.log_move("pump-001","A1","B2"); assert Path(CONFIG["EQUIPMENT_MOVES_LOG_PATH"]).exists()
q=t.make_qr("poctest"); assert isinstance(q,str) and len(q)>0
df_sop=t.sop_table(); print("SOP rows:", len(df_sop))
print("SMOKE_OK")

SOP rows: 3
SMOKE_OK


/tmp/ipykernel_36/2674203997.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[None]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  if mask.any(): df.loc[mask,:]=row.values
/tmp/ipykernel_36/2674203997.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[None]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  if mask.any(): df.loc[mask,:]=row.values


In [60]:
# --- [PATCH] UI launcher (Streamlit or ipywidgets fallback; guarded) ---
if CONFIG.get("RUN_UI", False):
    from IPython.display import display, clear_output

    # Minimal fallback UI (ipywidgets)
    import ipywidgets as W
    def _fallback_run_ui(tracker, get_state, get_actions, critic):
        box = W.Output(layout=W.Layout(border="1px solid #ddd", padding="6px"))
        btn_refresh = W.Button(description="Refresh")
        btn_actions = W.Button(description="Suggest Actions", button_style="info")

        def do_refresh(_):
            with box:
                clear_output()
                try:
                    state = get_state()
                    print("Operational dashboard (fallback UI)")
                    df = state.get("status_df") if isinstance(state, dict) else None
                    if df is not None:
                        try:
                            import pandas as pd  # optional
                            display(df.head())
                        except Exception:
                            print(df.head() if hasattr(df, "head") else df)
                    else:
                        print(state)
                except Exception as e:
                    print("state error:", e)

        def do_actions(_):
            with box:
                clear_output()
                try:
                    acts = get_actions()
                    print("Action candidates:")
                    print(acts)
                except Exception as e:
                    print("actions error:", e)

        btn_refresh.on_click(do_refresh)
        btn_actions.on_click(do_actions)
        display(W.VBox([W.HBox([btn_refresh, btn_actions]), box]))
        print("[ui] Streamlit missing → ipywidgets fallback active.")

    # Safe wrapper: try Streamlit UI, fall back if import fails anywhere
    def safe_run_ui(tracker, get_state, get_actions, critic):
        try:
            return run_ui(tracker=tracker, get_state=get_state, get_actions=get_actions, critic=critic)
        except ModuleNotFoundError as e:
            if getattr(e, "name", "") == "streamlit" or "streamlit" in str(e):
                return _fallback_run_ui(tracker, get_state, get_actions, critic)
            raise

    safe_run_ui(tracker=tracker, get_state=_get_state, get_actions=_get_actions, critic=TinyCritics())
else:
    print("UI disabled. Set CONFIG['RUN_UI']=True to launch.")


[ui] Streamlit missing → ipywidgets fallback active.


In [ ]:
try: CONFIG
except NameError: CONFIG = {}
CONFIG.setdefault("RUN_UI", False)
CONFIG.setdefault("RUN_PIPELINE", False)
print("[config]", CONFIG)


In [ ]:
EMBEDDED_CANONICAL_B64 = """..."""
EMBEDDED_CANONICAL_SHA256 = "8a54a37cbeb4d31ac543d0003964b10a9bf9a073ccb47bc41e37e3625af7e20a"
print("[embed] bytes:", len(EMBEDDED_CANONICAL_B64))


In [ ]:
import base64, json, types, sys
def _extract_py(payload: bytes) -> str:
    head = payload[:512]
    if head[:1]==b"{" and b'"cells"' in head:
        nb = json.loads(payload.decode("utf-8","replace"))
        out=[]
        for c in nb.get("cells", []):
            if c.get("cell_type")=="code":
                out.extend(c.get("source", [])); out.append("\n")
        code="".join(out)
    else:
        code = payload.decode("utf-8","replace")
    lines = code.splitlines()
    i=0;n=len(lines);lead=[]
    while i<n and (not lines[i].strip() or lines[i].lstrip().startswith("#")): lead.append(lines[i]); i+=1
    doc=[]; 
    def tri(s): s=s.strip(); return s.startswith('"""') or s.startswith("'''")
    if i<n and tri(lines[i]): 
        q=lines[i].strip()[:3]; doc.append(lines[i]); i+=1
        while i<n:
            doc.append(lines[i]); 
            if lines[i].strip().endswith(q): i+=1; break
            i+=1
    body=lines[i:]; fut=[]; rest=[]
    for L in body:
        (fut if L.lstrip().startswith("from __future__ import") else rest).append(L.strip() if L.lstrip().startswith("from __future__ import") else L)
    seen=set(); fut2=[]
    for L in fut:
        if L not in seen: fut2.append(L); seen.add(L)
    parts=[]; parts.extend(lead)
    if lead and lead[-1].strip(): parts.append("")
    parts.extend(doc)
    if doc and doc[-1].strip(): parts.append("")
    parts.extend(fut2)
    if fut2: parts.append("")
    parts.extend(rest)
    return "\n".join(parts)
def load_mod(name="ed_pipeline_v8"):
    payload = base64.b64decode(EMBEDDED_CANONICAL_B64)
    code = _extract_py(payload)
    m = types.ModuleType(name); m.__file__=f"<mem:{name}>"; sys.modules[name]=m
    exec(compile(code, m.__file__, "exec"), m.__dict__); return m
v8_mod = load_mod()
print("[assimilation]", all(hasattr(v8_mod, x) for x in ("WorkflowState","TinyCritics","CONFIG")))


In [ ]:
required=("WorkflowState","TinyCritics","CONFIG")
ok = all(hasattr(v8_mod, r) for r in required)
print("Core symbols present:", ok)
if not ok:
    raise RuntimeError("Hard gate failed: " + ", ".join([r for r in required if not hasattr(v8_mod, r)]))


In [ ]:
# --- [APPEND] Phase-2: process_hl7_and_log helper (guarded) ---
if CONFIG.get("RUN_PIPELINE", False):
    from typing import Dict, Any, Optional

    def process_hl7_and_log(hl7_text: str, *,
                            patient_id: str,
                            encounter_id: Optional[str] = None,
                            vitals_hint: Optional[Dict[str, Any]] = None,
                            labs_hint: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        """
        1) Extract troponin/D-dimer from HL7
        2) Compute Phase-2 bundle
        3) Append one-liners to event log
        Returns the bundle dict.
        """
        bundle = process_hl7_with_bundle(hl7_text, vitals_hint=vitals_hint, labs_hint=labs_hint)
        log_phase2_one_liners(bundle, patient_id=patient_id, encounter_id=encounter_id, source="hl7")
        return bundle

    # --- demo run (non-throwing) ---
    try:
        demo_hl7 = _demo_hl7  # from the HL7 adapter cell
    except NameError:
        demo_hl7 = """MSH|^~\\&|LAB||ED|||ORU^R01||
PID|1||12345^^^ED||DOE^JANE||19700101|F
OBR|1|||CHEST PAIN PANEL|||20250819010000
OBX|1|NM|TROP-HS^High Sens Troponin||60|ng/L|0-14||H|||F|||20250819020000
OBX|2|NM|TROP-HS^High Sens Troponin||30|ng/L|0-14||H|||F|||20250819010000
OBX|3|NM|D-DIMER^D-Dimer||0.9|mg/L|||||F|||20250819013000
"""
    demo_bundle = process_hl7_and_log(
        demo_hl7,
        patient_id="DEMO2", encounter_id="ED-0002",
        vitals_hint={"rr":16,"hr":105,"sbp":95,"temp_c":37.2,"gcs":14},
        labs_hint={"age":67, "risk_factors_count":3, "history":2, "ecg":1, "troponin_ratio":2.5,
                   "pao2_fio2":180,"platelets_x10e9_L":80,"bilirubin_mg_dL":3.0,"map_mmHg":65,"creatinine_mg_dL":2.2,"urine_ml_per_day":800,
                   "cardiac_arrest_at_admission":False,"st_deviation_present":True,"elevated_enzymes":True,"killip_class":2}
    )
    print("[phase2-pipeline] OK", " | ".join(demo_bundle["one_liners"]))
else:
    print("[phase2-pipeline] Skipped (CONFIG['RUN_PIPELINE']=False)")


In [ ]:
# --- [APPEND] Phase-2: HL7 Paste Panel (guarded UI) ---
if CONFIG.get("RUN_UI", False) and CONFIG.get("RUN_PIPELINE", False):
    import json
    import ipywidgets as W
    from IPython.display import display, clear_output

    ta_hl7 = W.Textarea(
        value="MSH|^~\\&|LAB||ED|||ORU^R01||\nPID|1||12345^^^ED||DOE^JANE||19700101|F\nOBR|1|||CHEST PAIN PANEL|||20250819010000\nOBX|1|NM|TROP-HS^High Sens Troponin||60|ng/L|0-14||H|||F|||20250819020000\nOBX|2|NM|TROP-HS^High Sens Troponin||30|ng/L|0-14||H|||F|||20250819010000\nOBX|3|NM|D-DIMER^D-Dimer||0.9|mg/L|||||F|||20250819013000\n",
        description="HL7", layout=W.Layout(width="100%", height="140px")
    )
    ti_pid = W.Text(value="DEMO", description="Patient ID")
    ti_enc = W.Text(value="ED-000X", description="Encounter ID")
    ta_vitals = W.Textarea(value='{"rr":16,"hr":105,"sbp":95,"temp_c":37.2,"gcs":14}',
                           description="Vitals JSON", layout=W.Layout(width="100%", height="70px"))
    ta_labs = W.Textarea(
        value='{"age":67,"risk_factors_count":3,"history":2,"ecg":1,"troponin_ratio":2.5,\n "pao2_fio2":180,"platelets_x10e9_L":80,"bilirubin_mg_dL":3.0,"map_mmHg":65,"creatinine_mg_dL":2.2,"urine_ml_per_day":800,\n "cardiac_arrest_at_admission":false,"st_deviation_present":true,"elevated_enzymes":true,"killip_class":2}',
        description="Labs JSON", layout=W.Layout(width="100%", height="100px")
    )
    btn_preview = W.Button(description="Preview (no log)")
    btn_log = W.Button(description="Compute + Log", button_style="success")
    out = W.Output(layout=W.Layout(border="1px solid #ddd", padding="6px"))

    def _parse_json(txt):
        try: return json.loads(txt) if txt.strip() else {}
        except Exception as e: return {"_parse_error": str(e)}

    def _render(bundle, logged=False):
        with out:
            clear_output()
            print("One-liners:")
            for line in bundle.get("one_liners", []):
                print(" •", line)
            print("\nScores:", bundle.get("scores", {}))
            print("Labs:", bundle.get("labs", {}))
            if logged: print("\nAppended to:", CONFIG.get("EVENT_LOG_PATH"))

    def on_preview(_):
        vit = _parse_json(ta_vitals.value); lab = _parse_json(ta_labs.value)
        bundle = process_hl7_with_bundle(ta_hl7.value, vitals_hint=vit, labs_hint=lab)
        _render(bundle, logged=False)

    def on_log(_):
        vit = _parse_json(ta_vitals.value); lab = _parse_json(ta_labs.value)
        bundle = process_hl7_and_log(ta_hl7.value, patient_id=ti_pid.value, encounter_id=ti_enc.value,
                                     vitals_hint=vit, labs_hint=lab)
        _render(bundle, logged=True)

    btn_preview.on_click(on_preview)
    btn_log.on_click(on_log)

    display(W.VBox([W.HBox([ti_pid, ti_enc]), ta_hl7, ta_vitals, ta_labs, W.HBox([btn_preview, btn_log]), out]))
    print("[phase2-ui] Paste panel ready.")
else:
    print("[phase2-ui] Skipped (set CONFIG['RUN_UI']=True and CONFIG['RUN_PIPELINE']=True)")


In [ ]:
# --- [APPEND] Phase-2: mount calculators/helpers from v7(1) (guarded) ---
if CONFIG.get("RUN_PIPELINE", False):
    import json
    from pathlib import Path

    mounted = {"from_v7_1": [], "wrappers": []}
    ERR = []

    v7p = Path("/mnt/data/ed_pipeline_v7(1).py")
    v7ns = {}
    if v7p.exists():
        try:
            code = v7p.read_text(encoding="utf-8", errors="replace")
            exec(compile(code, str(v7p), "exec"), v7ns)
            # calculators + lab helpers + result container
            for name in [
                "marburg_heart_score","heart_score","grace_points",
                "wells_pe","wells_dvt","sofa_score",
                "compute_age_adjusted_ddimer_threshold","interpret_ddimer",
                "compute_troponin_delta","ScoreResult","ResultsNotifier",
            ]:
                if name in v7ns:
                    globals()[name] = v7ns[name]
                    mounted["from_v7_1"].append(name)
        except Exception as e:
            ERR.append(f"v7(1) import failed: {e}")
    else:
        ERR.append("v7(1) missing")

    # Minimal fallbacks if not present in v7(1)
    if "qsofa" not in globals():
        def qsofa(rr, sbp, gcs):  # deterministic, minimal
            return int(rr >= 22) + int(sbp <= 100) + int(gcs < 15)
        mounted["wrappers"].append("qsofa(minimal)")
    if "mews" not in globals():
        def mews(rr, hr, sbp, temp_c, avpu="A"):
            s = 0
            s += 3 if rr <= 8 else 0 if rr <= 14 else 1 if rr <= 20 else 2 if rr <= 29 else 3
            s += 2 if hr <= 40 else 1 if hr <= 50 else 0 if hr <= 100 else 1 if hr <= 110 else 2 if hr <= 129 else 3
            s += 3 if sbp <= 70 else 2 if sbp <= 80 else 1 if sbp <= 100 else 0 if sbp <= 199 else 2
            s += 2 if temp_c < 35 else 0 if temp_c <= 38.4 else 2
            s += 0 if str(avpu).upper()=="A" else 3
            return s
        mounted["wrappers"].append("mews(minimal)")

    # Quick smoke checks (collect errors, don't crash)
    try:
        if "marburg_heart_score" in globals():
            assert marburg_heart_score(
                male_ge_55_or_female_ge_65=True, known_vascular_disease=True,
                pain_worse_with_exercise=False, patient_assumes_cardiac=True,
                pain_not_reproducible_by_palpation=False
            ).score >= 2
        if "heart_score" in globals():
            assert heart_score(history=2, ecg=1, age=2, risk_factors=2, troponin=1) >= 6
        if "grace_points" in globals():
            gp = grace_points(
                age=67, hr=110, sbp=95, creat_mgdl=1.6, killip_class=2,
                cardiac_arrest_at_admission=False, st_deviation_present=True, elevated_enzymes=True
            )
            cat = getattr(gp, "category", None) or (isinstance(gp, dict) and gp.get("category"))
            assert cat is None or cat.lower() in {"very high","high","intermediate"}
        if "sofa_score" in globals():
            s = sofa_score(
                pao2_fio2=180, platelets_x10e9_L=80, bilirubin_mg_dL=3.0, map_mmHg=65,
                vasopressors=None, gcs=13, creatinine_mg_dL=2.2, urine_ml_per_day=800
            )
            tot = getattr(s, "score", s if isinstance(s, (int,float)) else None)
            assert (tot is not None) and (tot >= 5)
        if "compute_age_adjusted_ddimer_threshold" in globals():
            assert 0.7 <= compute_age_adjusted_ddimer_threshold(70) <= 0.8
        if "qsofa" in globals(): assert qsofa(24, 95, 14) == 3
        if "mews"  in globals(): assert mews(rr=16, hr=105, sbp=95, temp_c=37.2, avpu="A") >= 2
    except AssertionError as e:
        ERR.append(f"smoke assert failed: {e}")

    print("[phase2-mount]", json.dumps({"mounted": mounted, "errors": ERR}, ensure_ascii=False))
else:
    print("[phase2-mount] Skipped (CONFIG['RUN_PIPELINE']=False)")


In [ ]:
# --- [APPEND] Phase-2 calculators imported from v7 + qSOFA/MEWS shims (guarded) ---
if CONFIG.get("RUN_PIPELINE", False):
    import json
    from pathlib import Path

    mounted = {"from_v7": [], "wrappers": []}
    ERR = []

    # Try multiple candidate locations; do NOT raise if missing
    candidate_paths = [
        Path("/mnt/data/ed_pipeline_v7(1).py"),
        Path("/mnt/data/ed_pipeline_v7.py"),
        Path("./ed_pipeline_v7(1).py"),
        Path("./ed_pipeline_v7.py"),
        Path("/kaggle/working/ed_pipeline_v7(1).py"),
        Path("/kaggle/working/ed_pipeline_v7.py"),
    ]

    v7_path = next((p for p in candidate_paths if p.exists()), None)
    v7ns = {}
    if v7_path:
        try:
            code = v7_path.read_text(encoding="utf-8", errors="replace")
            exec(compile(code, str(v7_path), "exec"), v7ns)
            for name in [
                "marburg_heart_score","heart_score","grace_points",
                "wells_pe","wells_dvt","sofa_score",
                "compute_age_adjusted_ddimer_threshold","interpret_ddimer",
                "compute_troponin_delta","ScoreResult","ResultsNotifier",
            ]:
                if name in v7ns:
                    globals()[name] = v7ns[name]
                    mounted["from_v7"].append(f"{name}@{v7_path.name}")
        except Exception as e:
            ERR.append(f"v7 import failed from {v7_path.name}: {e}")
    else:
        ERR.append("no v7 file found in candidate paths")

    # Minimal fallbacks if not present
    if "qsofa" not in globals():
        def qsofa(rr, sbp, gcs):  # deterministic, minimal
            return int(rr >= 22) + int(sbp <= 100) + int(gcs < 15)
        mounted["wrappers"].append("qsofa(minimal)")
    if "mews" not in globals():
        def mews(rr, hr, sbp, temp_c, avpu="A"):
            s = 0
            s += 3 if rr <= 8 else 0 if rr <= 14 else 1 if rr <= 20 else 2 if rr <= 29 else 3
            s += 2 if hr <= 40 else 1 if hr <= 50 else 0 if hr <= 100 else 1 if hr <= 110 else 2 if hr <= 129 else 3
            s += 3 if sbp <= 70 else 2 if sbp <= 80 else 1 if sbp <= 100 else 0 if sbp <= 199 else 2
            s += 2 if temp_c < 35 else 0 if temp_c <= 38.4 else 2
            s += 0 if str(avpu).upper()=="A" else 3
            return s
        mounted["wrappers"].append("mews(minimal)")

    # Quick smoke (non-fatal)
    try:
        if "qsofa" in globals():  assert qsofa(24, 95, 14) == 3
        if "mews"  in globals():  assert mews(rr=16, hr=105, sbp=95, temp_c=37.2, avpu="A") >= 2
        if "heart_score" in globals(): assert heart_score(history=2, ecg=1, age=2, risk_factors=2, troponin=1) >= 6
        if "sofa_score" in globals():
            s = sofa_score(
                pao2_fio2=180, platelets_x10e9_L=80, bilirubin_mg_dL=3.0, map_mmHg=65,
                vasopressors=None, gcs=13, creatinine_mg_dL=2.2, urine_ml_per_day=800
            )
            tot = getattr(s, "score", s if isinstance(s, (int,float)) else None)
            assert (tot is not None) and (tot >= 5)
    except AssertionError as e:
        ERR.append(f"smoke assert failed: {e}")

    print("[phase2-mount]", json.dumps({"mounted": mounted, "errors": ERR}, ensure_ascii=False))
else:
    print("[phase2-mount] Skipped (CONFIG['RUN_PIPELINE']=False)")


In [ ]:
# --- [APPEND] Phase-2: fallback calculators pack (guarded) ---
if CONFIG.get("RUN_PIPELINE", False):
    import math, typing

    # qSOFA (0..3)
    def qsofa(rr: float, sbp: float, gcs: float) -> int:
        return int(rr >= 22) + int(sbp <= 100) + int(gcs < 15)

    # MEWS (simplified bands)
    def mews(rr: float, hr: float, sbp: float, temp_c: float, avpu: str = "A") -> int:
        s = 0
        s += 3 if rr <= 8 else 0 if rr <= 14 else 1 if rr <= 20 else 2 if rr <= 29 else 3
        s += 2 if hr <= 40 else 1 if hr <= 50 else 0 if hr <= 100 else 1 if hr <= 110 else 2 if hr <= 129 else 3
        s += 3 if sbp <= 70 else 2 if sbp <= 80 else 1 if sbp <= 100 else 0 if sbp <= 199 else 2
        s += 2 if temp_c < 35 else 0 if temp_c <= 38.4 else 2
        s += 0 if str(avpu).upper() == "A" else 3
        return s

    # SOFA (minimal)
    class _SofaResult(typing.NamedTuple):
        score: int
    def sofa_score(*, pao2_fio2: float, platelets_x10e9_L: float, bilirubin_mg_dL: float,
                   map_mmHg: float, vasopressors: typing.Optional[bool],
                   gcs: int, creatinine_mg_dL: float, urine_ml_per_day: float):
        resp = 4 if pao2_fio2 < 100 else 3 if pao2_fio2 < 200 else 2 if pao2_fio2 < 300 else 1 if pao2_fio2 < 400 else 0
        coag = 4 if platelets_x10e9_L < 20 else 3 if platelets_x10e9_L < 50 else 2 if platelets_x10e9_L < 100 else 1 if platelets_x10e9_L < 150 else 0
        liver = 4 if bilirubin_mg_dL >= 12 else 3 if bilirubin_mg_dL >= 6 else 2 if bilirubin_mg_dL >= 2 else 1 if bilirubin_mg_dL >= 1.2 else 0
        cvs = 3 if vasopressors else (1 if map_mmHg < 70 else 0)
        cns = 4 if gcs < 6 else 3 if gcs < 10 else 2 if gcs < 13 else 1 if gcs < 15 else 0
        renal = 4 if (creatinine_mg_dL >= 5.0 or urine_ml_per_day < 200) else 3 if (creatinine_mg_dL >= 3.5 or urine_ml_per_day < 500) else 2 if creatinine_mg_dL >= 2.0 else 1 if creatinine_mg_dL >= 1.2 else 0
        return _SofaResult(resp + coag + liver + cvs + cns + renal)

    # HEART (minimal)
    def heart_score(*, history: int, ecg: int, age: int, risk_factors: int, troponin: int) -> int:
        a = 2 if age >= 65 else (1 if 45 <= age <= 64 else 0)
        return int(history) + int(ecg) + a + int(risk_factors) + int(troponin)

    # GRACE (coarse points → category)
    def grace_points(*, age: int, hr: int, sbp: int, creat_mgdl: float,
                     killip_class: int, cardiac_arrest_at_admission: bool,
                     st_deviation_present: bool, elevated_enzymes: bool):
        pts = 0
        pts += 1 if age >= 50 else 0; pts += 2 if age >= 70 else 0; pts += 2 if age >= 80 else 0
        pts += 1 if hr >= 100 else 0; pts += 2 if hr >= 110 else 0
        pts += 2 if sbp < 100 else 1 if sbp < 120 else 0
        pts += 1 if creat_mgdl >= 1.5 else 0
        pts += max(0, min(3, killip_class))
        pts += 2 if cardiac_arrest_at_admission else 0
        pts += 2 if st_deviation_present else 0
        pts += 1 if elevated_enzymes else 0
        cat = "very high" if pts >= 10 else "high" if pts >= 7 else "intermediate" if pts >= 4 else "low"
        return {"points": pts, "category": cat}

    # D-dimer thresholds
    def compute_age_adjusted_ddimer_threshold(age_years: int) -> float:
        if age_years <= 50: return 0.5
        decades_over = max(0, (age_years - 50) // 10)
        return 0.5 + 0.1 * decades_over

    def d_dimer_pregnancy_threshold(trimester: int) -> float:
        return {1: 0.7, 2: 1.0, 3: 1.3}.get(int(trimester), 0.7)

    def interpret_ddimer(value_mg_L: float, units: str, *, age_years: int = None, trimester: int = None):
        val = float(value_mg_L)
        th = None
        if age_years is not None: th = compute_age_adjusted_ddimer_threshold(int(age_years))
        if trimester is not None: th = max(th, d_dimer_pregnancy_threshold(int(trimester))) if th is not None else d_dimer_pregnancy_threshold(int(trimester))
        if th is None: th = 0.5
        return {"value": val, "threshold": th, "is_positive": val >= th}

    # Troponin operational delta
    def compute_troponin_delta(*, curr: float, prev: float, rel_pct_threshold: float = 20.0, abs_ng_L_threshold: float = 51.0):
        try:
            curr = float(curr); prev = float(prev)
        except Exception:
            return {"delta": None, "pct_change": None, "delta_flag": False}
        delta = curr - prev
        pct = (delta / prev * 100.0) if prev else None
        flag = (abs(delta) >= abs_ng_L_threshold) or ((pct is not None) and (abs(pct) >= rel_pct_threshold))
        return {"delta": delta, "pct_change": pct, "delta_flag": flag}

    # Smoke
    _errs = []
    try:
        assert qsofa(24, 95, 14) == 3
        assert mews(rr=16, hr=105, sbp=95, temp_c=37.2, avpu="A") >= 2
        assert heart_score(history=2, ecg=1, age=67, risk_factors=2, troponin=1) >= 6
        gp = grace_points(age=67, hr=110, sbp=95, creat_mgdl=1.6, killip_class=2,
                          cardiac_arrest_at_admission=False, st_deviation_present=True, elevated_enzymes=True)
        assert isinstance(gp, dict) and gp.get("category") in {"low","intermediate","high","very high"}
        s = sofa_score(pao2_fio2=180, platelets_x10e9_L=80, bilirubin_mg_dL=3.0, map_mmHg=65,
                       vasopressors=None, gcs=13, creatinine_mg_dL=2.2, urine_ml_per_day=800)
        assert hasattr(s, "score") and s.score >= 5
        assert 0.7 <= compute_age_adjusted_ddimer_threshold(70) <= 0.8
        td = compute_troponin_delta(curr=120, prev=60, rel_pct_threshold=20.0, abs_ng_L_threshold=51.0)
        assert td["delta_flag"] is True
    except AssertionError as e:
        _errs.append(str(e))
    print("[phase2-fallback] SMOKE_OK" if not _errs else f"[phase2-fallback] SMOKE_FAIL: {{_errs}}")
else:
    print("[phase2-fallback] Skipped (CONFIG['RUN_PIPELINE']=False)")


In [ ]:
# --- [APPEND] Phase-2: bundle helper (guarded) ---
if CONFIG.get("RUN_PIPELINE", False):
    from typing import Dict, Any

    def phase2_bundle(vitals: Dict[str, Any], labs: Dict[str, Any]) -> Dict[str, Any]:
        """
        Deterministic, low-cognitive-load wrapper that computes available Phase-2 scores.
        Inputs are dicts; missing keys simply skip that calculator.
        Returns a dict with structured results and short one-liners.
        """
        out = {"scores": {}, "labs": {}, "one_liners": []}

        # qSOFA
        if all(k in vitals for k in ("rr","sbp","gcs")):
            qs = qsofa(vitals["rr"], vitals["sbp"], vitals["gcs"])
            out["scores"]["qsofa"] = qs
            out["one_liners"].append(f"qSOFA: {qs}")

        # MEWS
        if all(k in vitals for k in ("rr","hr","sbp","temp_c")):
            avpu = vitals.get("avpu","A")
            mw = mews(vitals["rr"], vitals["hr"], vitals["sbp"], vitals["temp_c"], avpu=avpu)
            out["scores"]["mews"] = mw
            out["one_liners"].append(f"MEWS: {mw}")

        # HEART (minimal)
        # expect: history (0..2), ecg (0..2), age (years), risk_factors_count (int), troponin_ratio (float)
        if all(k in labs for k in ("history","ecg","age","risk_factors_count","troponin_ratio")):
            rf = labs["risk_factors_count"]
            rf_pts = 2 if rf >= 3 else (1 if rf >= 1 else 0)
            t_ratio = labs["troponin_ratio"]
            t_pts = 2 if t_ratio >= 3 else (1 if t_ratio > 1 else 0)
            hs = heart_score(history=labs["history"], ecg=labs["ecg"], age=labs["age"],
                             risk_factors=rf_pts, troponin=t_pts)
            out["scores"]["heart"] = hs
            out["one_liners"].append(f"HEART: {hs}")

        # GRACE (coarse)
        gp_keys = ("age","hr","sbp","creat_mgdl","killip_class",
                   "cardiac_arrest_at_admission","st_deviation_present","elevated_enzymes")
        if all(k in labs for k in gp_keys):
            gp = grace_points(age=labs["age"], hr=labs["hr"], sbp=labs["sbp"],
                              creat_mgdl=labs["creat_mgdl"], killip_class=labs["killip_class"],
                              cardiac_arrest_at_admission=labs["cardiac_arrest_at_admission"],
                              st_deviation_present=labs["st_deviation_present"],
                              elevated_enzymes=labs["elevated_enzymes"])
            out["scores"]["grace"] = gp
            cat = gp["category"] if isinstance(gp, dict) else getattr(gp, "category", None)
            out["one_liners"].append(f"GRACE: {cat or gp}")

        # SOFA (minimal)
        sf_keys = ("pao2_fio2","platelets_x10e9_L","bilirubin_mg_dL","map_mmHg","gcs","creatinine_mg_dL","urine_ml_per_day")
        if all(k in labs for k in sf_keys):
            sf = sofa_score(pao2_fio2=labs["pao2_fio2"], platelets_x10e9_L=labs["platelets_x10e9_L"],
                            bilirubin_mg_dL=labs["bilirubin_mg_dL"], map_mmHg=labs["map_mmHg"],
                            vasopressors=labs.get("vasopressors"), gcs=labs["gcs"],
                            creatinine_mg_dL=labs["creatinine_mg_dL"], urine_ml_per_day=labs["urine_ml_per_day"])
            total = getattr(sf, "score", sf if isinstance(sf, (int,float)) else None)
            out["scores"]["sofa"] = total
            out["one_liners"].append(f"SOFA: {total}")

        # D-dimer
        if "ddimer_mg_L" in labs:
            th = None
            if "age" in labs:
                th = compute_age_adjusted_ddimer_threshold(labs["age"])
            if "pregnancy_trimester" in labs:
                bump = d_dimer_pregnancy_threshold(labs["pregnancy_trimester"])
                th = max(th, bump) if th is not None else bump
            th = th if th is not None else 0.5
            is_pos = labs["ddimer_mg_L"] >= th
            out["labs"]["ddimer"] = {"value": labs["ddimer_mg_L"], "threshold": th, "is_positive": is_pos}
            out["one_liners"].append(f"D-dimer: {labs['ddimer_mg_L']} mg/L (cutoff {th}) -> {'POS' if is_pos else 'NEG'}")

        # Troponin delta (operational)
        if all(k in labs for k in ("troponin_curr_ng_L","troponin_prev_ng_L")):
            td = compute_troponin_delta(curr=labs["troponin_curr_ng_L"], prev=labs["troponin_prev_ng_L"],
                                        rel_pct_threshold=labs.get("troponin_rel_pct_threshold", 20.0),
                                        abs_ng_L_threshold=labs.get("troponin_abs_ng_L_threshold", 51.0))
            out["labs"]["troponin_delta"] = td
            if td["delta"] is not None:
                d = f"{td['delta']:+.0f}"
                p = "" if td["pct_change"] is None else f", {td['pct_change']:+.0f}%"
                out["one_liners"].append(f"Troponin Δ: {d}{p} -> {'CHECK' if td['delta_flag'] else 'OK'}")

        return out

    # smoke: non-throwing demo
    _demo = phase2_bundle(
        vitals={"rr":16,"hr":105,"sbp":95,"temp_c":37.2,"gcs":14,"avpu":"A"},
        labs={
            "history":2,"ecg":1,"age":67,"risk_factors_count":3,"troponin_ratio":2.5,
            "pao2_fio2":180,"platelets_x10e9_L":80,"bilirubin_mg_dL":3.0,"map_mmHg":65,"gcs":13,"creatinine_mg_dL":2.2,"urine_ml_per_day":800,
            "ddimer_mg_L":0.9,"pregnancy_trimester":1,
            "troponin_curr_ng_L":120,"troponin_prev_ng_L":60
        }
    )
    print("[phase2-bundle] SMOKE_OK", " | ".join(_demo["one_liners"]))
else:
    print("[phase2-bundle] Skipped (CONFIG['RUN_PIPELINE']=False)")


In [ ]:
# Backfill CONFIG defaults needed for logging (append-only, non-destructive)
try:
    CONFIG
except NameError:
    CONFIG = {}

CONFIG.setdefault("DATA_ROOT", "/mnt/data")
CONFIG.setdefault("EVENT_LOG_PATH", f"{CONFIG['DATA_ROOT'].rstrip('/')}/event_log.jsonl")

print("EVENT_LOG_PATH:", CONFIG["EVENT_LOG_PATH"])


In [ ]:
# --- [APPEND] Phase-2: event-log sink (guarded) ---
if CONFIG.get("RUN_PIPELINE", False):
    from datetime import datetime, timezone
    import json, os

    def _append_event(event: dict, path: str):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, ensure_ascii=False) + "\n")

    def log_phase2_one_liners(bundle: dict, *, patient_id="UNKNOWN", encounter_id=None, source="phase2"):
        """
        Append each one-liner from phase2_bundle to CONFIG['EVENT_LOG_PATH'] as JSONL.
        Non-throwing; prints a short summary.
        """
        now = datetime.now(timezone.utc).isoformat()
        lines = bundle.get("one_liners", [])
        for idx, line in enumerate(lines, 1):
            evt = {
                "ts": now,
                "type": "phase2.one_liner",
                "patient_id": patient_id,
                "encounter_id": encounter_id,
                "seq": idx,
                "text": line,
                "scores": bundle.get("scores", {}),
                "labs": bundle.get("labs", {}),
                "source": source,
            }
            _append_event(evt, CONFIG["EVENT_LOG_PATH"])
        print(f"[phase2-log] wrote {len(lines)} lines to", CONFIG["EVENT_LOG_PATH"])

    # --- demo write (uses prior _bundle if available, else synthesizes a small one) ---
    try:
        demo_bundle = _bundle  # from the HL7 adapter demo
    except NameError:
        demo_bundle = phase2_bundle(
            vitals={"rr":16,"hr":105,"sbp":95,"temp_c":37.2,"gcs":14},
            labs={
                "history":2,"ecg":1,"age":67,"risk_factors_count":3,"troponin_ratio":2.5,
                "pao2_fio2":180,"platelets_x10e9_L":80,"bilirubin_mg_dL":3.0,"map_mmHg":65,"creatinine_mg_dL":2.2,"urine_ml_per_day":800,
                "ddimer_mg_L":0.9,"troponin_curr_ng_L":60,"troponin_prev_ng_L":30,
                "cardiac_arrest_at_admission":False,"st_deviation_present":True,"elevated_enzymes":True,"killip_class":2
            }
        )
    log_phase2_one_liners(demo_bundle, patient_id="DEMO", encounter_id="ED-0001")
else:
    print("[phase2-log] Skipped (CONFIG['RUN_PIPELINE']=False)")


In [ ]:
# --- [APPEND] Phase-2: supplement from Phase-2 notebooks (guarded) ---
if CONFIG.get("RUN_PIPELINE", False):
    import json
    from pathlib import Path

    mounted = {"from_scores_nb": [], "from_phase23_nb": [], "wrappers": []}
    ERR = []

    # 1) Calculators from ED_Phase_2_scores_v1(3).ipynb
    scores_nb = Path("/mnt/data/ED_Phase_2_scores_v1(3).ipynb")
    if scores_nb.exists():
        try:
            nb = json.loads(scores_nb.read_text(encoding="utf-8", errors="replace"))
            src = "\n\n".join(
                "".join(c.get("source", []))
                for c in nb.get("cells", []) if c.get("cell_type") == "code"
            )
            ns = {}
            exec(compile(src, str(scores_nb), "exec"), ns)

            # calculators (bind or upgrade)
            if "heart_score" in ns: globals()["heart_score"] = ns["heart_score"]; mounted["from_scores_nb"].append("heart_score")
            if "marburg_heart_score" in ns: globals()["marburg_heart_score"] = ns["marburg_heart_score"]; mounted["from_scores_nb"].append("marburg_heart_score")
            if "qsofa" in ns: globals()["qsofa"] = ns["qsofa"]; mounted["from_scores_nb"].append("qsofa")
            if "mews" in ns:  globals()["mews"]  = ns["mews"];  mounted["from_scores_nb"].append("mews")

            # GRACE naming: grace_inhospital_points -> grace_points
            if "grace_inhospital_points" in ns:
                def grace_points(**kw): return ns["grace_inhospital_points"](**kw)
                globals()["grace_points"] = grace_points
                mounted["wrappers"].append("grace_points->grace_inhospital_points")

            # SOFA naming: sofa -> sofa_score
            if "sofa" in ns:
                def sofa_score(**kw): return ns["sofa"](**kw)
                globals()["sofa_score"] = sofa_score
                mounted["wrappers"].append("sofa_score->sofa")

            # D-dimer thresholds
            if "d_dimer_age_adjusted_threshold" in ns:
                globals()["compute_age_adjusted_ddimer_threshold"] = ns["d_dimer_age_adjusted_threshold"]
                mounted["from_scores_nb"].append("compute_age_adjusted_ddimer_threshold")
            if "d_dimer_pregnancy_threshold" in ns:
                globals()["d_dimer_pregnancy_threshold"] = ns["d_dimer_pregnancy_threshold"]
                mounted["from_scores_nb"].append("d_dimer_pregnancy_threshold")
        except Exception as e:
            ERR.append(f"scores_nb import failed: {e}")
    else:
        ERR.append("scores_nb missing")

    # 2) HL7 + troponin helpers from ED-Pipeline-Phase2-phase23(1).ipynb
    phase23_nb = Path("/mnt/data/ED-Pipeline-Phase2-phase23(1).ipynb")
    if phase23_nb.exists():
        try:
            nb = json.loads(phase23_nb.read_text(encoding="utf-8", errors="replace"))
            src = "\n\n".join(
                "".join(c.get("source", []))
                for c in nb.get("cells", []) if c.get("cell_type") == "code"
            )
            ns2 = {}
            exec(compile(src, str(phase23_nb), "exec"), ns2)

            for name in [
                "extract_troponins_from_hl7",
                "compute_troponin_delta_from_hl7",
                "pipeline_step_process_hl7_troponin",
                "recommend_troponin_actions",
                "schedule_serial_troponin",
                "ddimer_adjusted_cutoff",
                "ddimer_decision",
            ]:
                if name in ns2:
                    globals()[name] = ns2[name]
                    mounted["from_phase23_nb"].append(name)
        except Exception as e:
            ERR.append(f"phase23_nb import failed: {e}")
    else:
        ERR.append("phase23_nb missing")

    # 3) Quick smoke (collect errors, don't crash)
    try:
        if "qsofa" in globals():  assert qsofa(24, 95, 14) == 3
        if "mews"  in globals():  assert mews(rr=16, hr=105, sbp=95, temp_c=37.2, avpu="A") >= 2
        if "heart_score" in globals(): assert heart_score(history=2, ecg=1, age=2, risk_factors=2, troponin=1) >= 6
        if "marburg_heart_score" in globals():
            r = marburg_heart_score(
                male_ge_55_or_female_ge_65=True, known_vascular_disease=True,
                pain_worse_with_exercise=False, patient_assumes_cardiac=True,
                pain_not_reproducible_by_palpation=False
            )
            assert getattr(r, "score", 2) >= 2
        if "grace_points" in globals():
            gp = grace_points(
                age=67, hr=110, sbp=95, creat_mgdl=1.6, killip_class=2,
                cardiac_arrest_at_admission=False, st_deviation_present=True, elevated_enzymes=True
            )
            cat = getattr(gp, "category", None) or (isinstance(gp, dict) and gp.get("category"))
            assert (cat is None) or (str(cat).lower() in {"very high","high","intermediate"})
        if "sofa_score" in globals():
            s = sofa_score(
                pao2_fio2=180, platelets_x10e9_L=80, bilirubin_mg_dL=3.0, map_mmHg=65,
                vasopressors=None, gcs=13, creatinine_mg_dL=2.2, urine_ml_per_day=800
            )
            tot = getattr(s, "score", s if isinstance(s, (int,float)) else None)
            assert (tot is not None) and (tot >= 5)
        if "compute_age_adjusted_ddimer_threshold" in globals():
            assert 0.7 <= compute_age_adjusted_ddimer_threshold(70) <= 0.8
    except AssertionError as e:
        ERR.append(f"smoke assert failed: {e}")

    print("[phase2-supplement]", json.dumps({"mounted": mounted, "errors": ERR}, ensure_ascii=False))
else:
    print("[phase2-supplement] Skipped (CONFIG['RUN_PIPELINE']=False)")


In [ ]:
# Tail the last 20 Phase-2 entries in the JSONL event log
import json, pathlib
p = pathlib.Path(CONFIG["EVENT_LOG_PATH"])
lines = p.read_text(encoding="utf-8").splitlines()
for raw in lines[-20:]:
    evt = json.loads(raw)
    if evt.get("type") == "phase2.one_liner":
        print(evt["ts"], "-", evt["text"])


In [ ]:
print("Has bundle:", "phase2_bundle" in globals())
print("Has HL7 helper:", "process_hl7_and_log" in globals())
